In [7]:
import geopandas as gpd
import pandas as pd

In [2]:
# 침수흔적도 데이터
flood = gpd.read_file("C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/침수흔적도/침수흔적도_2025.shp")
flood2 = gpd.read_file("C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/침수흔적도/침수흔적도_2024.shp")
flood3 = gpd.read_file("C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/침수흔적도/침수흔적도_2023.shp")

In [3]:
# 서울시 법정동 경계 데이터 로드 및 좌표계 변환
seoul_shp = "C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/행정구역/행정구역_법정동_경계/admstr_zone_lgldong_bndry_24.shp"
gdf = gpd.read_file(seoul_shp)

In [ ]:
# 시각화를 위해 위경도(EPSG:4326)로 변환
if gdf.crs != 'EPSG:4326':
    gdf = gdf.to_crs(epsg = 4326)

In [ ]:
# 정확한 면적 계산을 위해 미터 단위 투영 좌표계(EPSG:5179)로 임시 변환하여 각 동의 전체 면적 구하기
gdf['dong_area'] = gdf.to_crs(epsg = 5179).geometry.area

In [ ]:
# 좌표계 통일
for f in [flood, flood2, flood3]:
    if f.crs != 'EPSG:4326':
        f.to_crs(epsg = 4326, inplace = True)

In [ ]:
# 3개년 데이터를 하나로 행 병합
flood_combined = pd.concat([flood, flood2, flood3], ignore_index = True) 

In [ ]:
# 동일한 위치가 여러 해에 걸쳐 침수되었을 경우를 대비해 겹치는 Polygon을 하나로 Union
flood_combined['dummy'] = 1  # 병합을 위한 임시 컬럼 생성
flood_combined = flood_combined.dissolve(by = 'dummy').reset_index()

In [ ]:
# 동 경계와 합쳐진 '3개년 통합 침수흔적도' 겹치는 부분 추출
intersection = gpd.overlay(gdf, flood_combined, how = 'intersection')

In [11]:
# 통합 침수된 영역의 실제 면적 계산 (미터 단위로 계산)
intersection['flood_area'] = intersection.to_crs(epsg=5179).geometry.area

In [12]:
# 법정동별 침수 면적 집계 및 병합
dong_col = 'EMD_CD' 

flood_sum = intersection.groupby(dong_col)['flood_area'].sum().reset_index()

In [13]:
# 기존 동 경계 데이터에 침수 면적 합계를 병합
merged_gdf = gdf.merge(flood_sum, on=dong_col, how='left')

# 침수되지 않은 동은 NaN이 되므로 0으로 채움
merged_gdf['flood_area'] = merged_gdf['flood_area'].fillna(0)

In [14]:
# 3개년 통합 침수 비율(%) 계산
merged_gdf['통합_침수비율'] = (merged_gdf['flood_area'] / merged_gdf['dong_area']) * 100

In [15]:
# 동별 통합 침수 비율 시각화
m_flood = merged_gdf.explore(
    column='통합_침수비율',                         
    cmap='Blues',                            
    tooltip=['EMD_NM', '통합_침수비율'],           
    style_kwds={
        'weight': 0.5,                       
        'color': 'gray',                     
        'fillOpacity': 0.75                  
    },
    tiles='CartoDB positron',                
    name='3-Year Flood Ratio by Dong'
)

In [16]:
m_flood.save("seoul_dong_3year_flood_ratio.html")